In [19]:
import pandas as pd
import networkx as nx

# 1. Initialize the Graph
cairo_network = nx.Graph()

# We create a dictionary to easily look up names by ID later
id_to_name = {}

# 2. Load the Neighborhoods
print("Loading Neighborhoods...")
df_neighborhoods = pd.read_csv("Geographic_Data(Neighborhoods_and_Districts).csv")

for index, row in df_neighborhoods.iterrows():
    node_id = str(row['ID']) # FORCE TO STRING
    id_to_name[node_id] = row['Name']

    cairo_network.add_node(
        node_id,
        name=row['Name'],
        population=row['Population'],
        type=row['Type']
    )

# 3. Load Important Facilities (This handles F1, F2, etc.)
print("Loading Important Facilities...")
df_facilities = pd.read_csv("Geographic_Data(Important_Facilities).csv")

for index, row in df_facilities.iterrows():
    # THE FIX: Changed 'Facility_ID' to 'ID'
    facility_id = str(row['ID']) # FORCE TO STRING
    id_to_name[facility_id] = row['Name']

    cairo_network.add_node(
        facility_id,
        name=row['Name'],
        type=row['Type']
    )

# 4. Load the Existing Roads
print("Loading Existing Roads...")
df_existing_roads = pd.read_csv("Road_Network_Data(Existing_Roads).csv")

for index, row in df_existing_roads.iterrows():
    from_id = str(row['FromID']) # FORCE TO STRING
    to_id = str(row['TOID'])     # FORCE TO STRING

    cairo_network.add_edge(
        from_id,
        to_id,
        weight=row['Distance(km)'],
        capacity=row['Current Capacity(vehicles/hour)'],
        condition=row['Condition(1-10)']
    )

print(f"Graph completely rebuilt! We have {cairo_network.number_of_nodes()} total locations and {cairo_network.number_of_edges()} existing roads.")

Loading Neighborhoods...
Loading Important Facilities...
Loading Existing Roads...
Graph completely rebuilt! We have 25 total locations and 27 existing roads.


In [20]:
# 1. Implement Route Planning (Dijkstra's Algorithm)
def find_shortest_path(graph, start_id, end_id):
    print(f"Calculating optimal route from {id_to_name[start_id]} to {id_to_name[end_id]}...")
    try:
        # nx.shortest_path uses Dijkstra's under the hood to find the optimal route
        path = nx.shortest_path(graph, source=start_id, target=end_id, weight='weight')

        # nx.shortest_path_length calculates the total cost (in this case, distance in km)
        total_distance = nx.shortest_path_length(graph, source=start_id, target=end_id, weight='weight')

        # Convert the list of numerical IDs back to actual Neighborhood names
        path_names = [id_to_name[node_id] for node_id in path]

        print(f"Optimal Route: {' -> '.join(path_names)}")
        print(f"Total Distance: {total_distance} km")

    except nx.NetworkXNoPath:
        print("Error: No valid path exists between these two locations yet.")
    except KeyError:
        print("Error: One or both of these Neighborhood IDs do not exist in the data.")

# 2. Let's test it!
# Pick two valid IDs from your Geographic_Data(Neighborhoods_and_Districts).csv
test_start_id = '1'
test_destination_id = '5'

find_shortest_path(cairo_network, start_id=test_start_id, end_id=test_destination_id)

Calculating optimal route from Maadi to Heliopolis...
Optimal Route: Maadi -> Downtown Cairo -> Heliopolis
Total Distance: 14.6 km


In [24]:
print(f"Total edges in cairo_network right now: {cairo_network.number_of_edges()}")

Total edges in cairo_network right now: 27


In [25]:
import pandas as pd
import networkx as nx

# 1. Create a new graph specifically for Infrastructure Planning
planning_network = nx.Graph()

# 2. Add all nodes from our existing city map
planning_network.add_nodes_from(cairo_network.nodes(data=True))

# 3. Add Existing Roads (Using Distance as the weight)
for u, v, data in cairo_network.edges(data=True):
    planning_network.add_edge(u, v, weight=data['weight'], cost=0, type='existing')

# 4. Load and Add Potential New Roads (Using Distance as the weight)
print("Loading Potential New Roads...")
df_potential = pd.read_csv("Road_Network_Data(Potential_New_Roads).csv")

for index, row in df_potential.iterrows():
    planning_network.add_edge(
        str(row['FromID']), # FORCE TO STRING
        str(row['TOID']),   # FORCE TO STRING
        weight=row['Distance(km)'],
        cost=row['Construction Cost(Million EGP)'],
        type='potential'
    )

# 5. Run the Minimum Spanning Tree Algorithm
print("Running Distance-Optimized MST Algorithm...")
mst_network = nx.minimum_spanning_tree(planning_network, weight='weight')

# 6. Analyze the Results
new_roads_to_build = [(u, v, data) for u, v, data in mst_network.edges(data=True) if data['type'] == 'potential']
total_cost = sum(data['cost'] for u, v, data in new_roads_to_build)

print("\n--- NEW INFRASTRUCTURE MASTER PLAN ---")
if len(new_roads_to_build) > 0:
    print(f"To create the most efficient network, you must build {len(new_roads_to_build)} new strategic roads.")
    print(f"Total Budget Required: {total_cost} Million EGP")
    print("\nPriority Roads to Construct:")
    for u, v, data in new_roads_to_build:
        # THE FIX: Use str() instead of int()
        print(f"- Connect {id_to_name[str(u)]} to {id_to_name[str(v)]} (Cost: {data['cost']}M EGP, Distance: {data['weight']} km)")
else:
    print("The existing network is already the shortest possible spanning tree!")

# 7. Upgrade the Main City Network
for u, v, data in new_roads_to_build:
    # THE FIX: Use str() instead of int() here as well
    cairo_network.add_edge(str(u), str(v), weight=data['weight'], capacity=4000, condition=10)

print(f"\nMain network upgraded! Cairo now has {cairo_network.number_of_edges()} total roads.")

Loading Potential New Roads...
Running Distance-Optimized MST Algorithm...

--- NEW INFRASTRUCTURE MASTER PLAN ---
To create the most efficient network, you must build 1 new strategic roads.
Total Budget Required: 450 Million EGP

Priority Roads to Construct:
- Connect Mohandessin to Sheikh Zayed (Cost: 450M EGP, Distance: 22.7 km)

Main network upgraded! Cairo now has 28 total roads.


In [26]:
# 1. Load Traffic Flow Data
print("Loading Traffic Patterns...")
df_traffic = pd.read_csv("Traffic_Flow_Data_Patterns.csv")

# Create a dictionary to quickly look up traffic by RoadID (e.g., '1-3')
traffic_data = {}
for index, row in df_traffic.iterrows():
    traffic_data[row['RoadID']] = {
        'Morning': row['Morning Peak(veh/h)'],
        'Afternoon': row['Afternoon (veh/h)'],
        'Evening': row['Evening Peak(veh/h)'],
        'Night': row['Night(veh/h)']
    }

# 2. Implement Time-Aware Dijkstra
def find_dynamic_route(graph, start_id, end_id, time_of_day):
    print(f"\n--- Calculating {time_of_day} Route from {id_to_name[start_id]} to {id_to_name[end_id]} ---")

    # Create a temporary copy of the graph so we don't mess up our base map
    dynamic_graph = graph.copy()

    # 3. Adjust Road Weights based on Traffic
    for u, v, data in dynamic_graph.edges(data=True):
        # Construct the RoadID exactly how it looks in the CSV (e.g., '1-3' or '3-1')
        road_id_1 = f"{u}-{v}"
        road_id_2 = f"{v}-{u}"

       # traffic_volume = 0
        # Find the traffic data for this specific road
       # if road_id_1 in traffic_data:
        #    traffic_volume = traffic_data[road_id_1][time_of_day]
        #elif road_id_2 in traffic_data:
         #   traffic_volume = traffic_data[road_id_2][time_of_day]


       # We ask the newly trained AI model for the traffic forecast!
        traffic_volume = predict_congestion(road_id_1, time_of_day)

        # 4. The Algorithm Math:
        # Base travel time is roughly distance / speed (let's assume 60 km/h base speed -> 1 km takes 1 minute)
        base_time_minutes = data['weight'] * 1.0

        # If traffic volume is high (e.g., > 2000 vehicles), we add a delay multiplier
        # This simulates severe congestion!
        traffic_multiplier = 1.0 + (traffic_volume / 1500)

        # Calculate the final dynamic weight
        dynamic_time = base_time_minutes * traffic_multiplier

        # Update the edge in our temporary graph
        dynamic_graph[u][v]['dynamic_time'] = dynamic_time

    # 5. Run Dijkstra using our new 'dynamic_time' instead of physical distance
    try:
        path = nx.shortest_path(dynamic_graph, source=start_id, target=end_id, weight='dynamic_time')
        total_time = nx.shortest_path_length(dynamic_graph, source=start_id, target=end_id, weight='dynamic_time')

        path_names = [id_to_name[node_id] for node_id in path]
        print(f"Optimal Route: {' -> '.join(path_names)}")
        print(f"Estimated Travel Time: {round(total_time, 1)} minutes")

    except nx.NetworkXNoPath:
        print("Error: No valid path exists.")
    except KeyError:
        print("Error: Invalid Neighborhood ID.")

# 6. Let's test how traffic changes the route!
test_start = '1'  # Maadi
test_end = '5'    # Heliopolis

# Compare Night (Empty roads) vs Evening (Rush Hour)
find_dynamic_route(cairo_network, start_id=test_start, end_id=test_end, time_of_day='Night')
find_dynamic_route(cairo_network, start_id=test_start, end_id=test_end, time_of_day='Evening')

Loading Traffic Patterns...

--- Calculating Night Route from Maadi to Heliopolis ---
Optimal Route: Maadi -> Downtown Cairo -> Heliopolis
Estimated Travel Time: 29.2 minutes

--- Calculating Evening Route from Maadi to Heliopolis ---
Optimal Route: Maadi -> Downtown Cairo -> Heliopolis
Estimated Travel Time: 29.2 minutes


In [ ]:
import math

# 1. Inject Coordinates into our Graph
print("Adding GPS Coordinates for Emergency Routing...")
df_neighborhoods = pd.read_csv("Geographic_Data(Neighborhoods_and_Districts).csv")
for index, row in df_neighborhoods.iterrows():
    node_id = str(row['ID'])
    # Add X, Y coordinates to the existing nodes
    cairo_network.nodes[node_id]['x'] = row['X-coordinate']
    cairo_network.nodes[node_id]['y'] = row['Y-coordinate']

df_facilities = pd.read_csv("Geographic_Data(Important_Facilities).csv")
for index, row in df_facilities.iterrows():
    facility_id = str(row['ID'])
    cairo_network.nodes[facility_id]['x'] = row['X-coordinate']
    cairo_network.nodes[facility_id]['y'] = row['Y-coordinate']

print("Coordinates successfully added!")

Adding GPS Coordinates for Emergency Routing...
Coordinates successfully added!


In [ ]:
# 2. Define the Heuristic (Straight-line distance)
def calculate_straight_line(node1, node2, graph):
    try:
        x1, y1 = graph.nodes[node1]['x'], graph.nodes[node1]['y']
        x2, y2 = graph.nodes[node2]['x'], graph.nodes[node2]['y']

        # Euclidean distance formula
        # Multiplied by roughly 111 to convert map degrees to kilometers
        return math.sqrt((x2 - x1)**2 + (y2 - y1)**2) * 111
    except KeyError:
        return 0 # Fallback if coordinates are missing

# 3. Implement the A* Search Algorithm
def route_ambulance(graph, start_id, end_id):
    print(f"\n🚨 EMERGENCY ROUTING INITIATED: {id_to_name[start_id]} to {id_to_name[end_id]} 🚨")
    try:
        # We use nx.astar_path and pass it our custom heuristic function
        path = nx.astar_path(
            graph,
            source=start_id,
            target=end_id,
            heuristic=lambda u, v: calculate_straight_line(u, v, graph),
            weight='weight' # For an ambulance with sirens, we assume they bypass normal traffic!
        )

        total_distance = nx.shortest_path_length(graph, source=start_id, target=end_id, weight='weight')
        path_names = [id_to_name[node_id] for node_id in path]

        print(f"Fastest Emergency Route: {' -> '.join(path_names)}")
        print(f"Total Distance: {total_distance} km")
        print("Note: Sirens active. Traffic patterns ignored.")

    except nx.NetworkXNoPath:
        print("CRITICAL ERROR: No path to facility!")

# 4. Let's Test an Emergency!
# Let's route an ambulance from Downtown (ID '3') to a Hospital.
# Looking at your facilities data, 'F1' or 'F2' are likely medical or emergency centers.
test_emergency_start = '3'
test_hospital_target = 'F1'

route_ambulance(cairo_network, start_id=test_emergency_start, end_id=test_hospital_target)


🚨 EMERGENCY ROUTING INITIATED: Downtown Cairo to Cairo International Airport 🚨
Fastest Emergency Route: Downtown Cairo -> Nasr City -> Cairo International Airport
Total Distance: 13.6 km
Note: Sirens active. Traffic patterns ignored.


In [ ]:
import pandas as pd
import numpy as np

# 1. Load Public Transit Demand Data
print("Loading Public Transit Demand Data...")
try:
    df_transit = pd.read_csv("Public_Transportation_Data(demand).csv")
    # Assuming columns like: 'Route_ID', 'Buses_Required', 'Passenger_Demand'
    routes = df_transit['Route_ID'].tolist()
    buses_required = df_transit['Buses_Required'].tolist()
    passenger_demand = df_transit['Passenger_Demand'].tolist()
except Exception as e:
    print("Warning: CSV column names might differ. Using simulated Cairo data for the DP model...")
    # Fallback simulated data based on Cairo's major arteries
    routes = ['Route 1 (Ring Road)', 'Route 2 (Autostrad)', 'Route 3 (Salah Salem)', 'Route 4 (Corniche)', 'Route 5 (90th St)']
    buses_required = [15, 10, 12, 8, 5]     # Cost (Weight)
    passenger_demand = [8000, 5000, 6500, 4000, 2500] # Value (Passengers served)

# 2. Set our Fleet Limit
TOTAL_AVAILABLE_BUSES = 30

# 3. Implement Dynamic Programming (0/1 Knapsack Algorithm)
def optimize_fleet_allocation(routes, cost, value, total_budget):
    n = len(routes)

    # Create a 2D array (our "memoization" table) filled with zeros
    # dp_table[i][w] will store the maximum passengers served using the first 'i' routes and 'w' buses
    dp_table = [[0 for _ in range(total_budget + 1)] for _ in range(n + 1)]

    # Build the DP table bottom-up
    for i in range(1, n + 1):
        for w in range(1, total_budget + 1):
            if cost[i-1] <= w:
                # Choice 1: We assign buses to this route + whatever is best for the remaining buses
                include_route = value[i-1] + dp_table[i-1][w - cost[i-1]]
                # Choice 2: We DO NOT assign buses to this route
                exclude_route = dp_table[i-1][w]

                # Save the maximum of the two choices
                dp_table[i][w] = max(include_route, exclude_route)
            else:
                # We don't have enough buses for this route, so we exclude it
                dp_table[i][w] = dp_table[i-1][w]

    # 4. Trace back to find which exact routes were selected
    max_passengers = dp_table[n][total_budget]
    w = total_budget
    selected_routes = []

    for i in range(n, 0, -1):
        if max_passengers <= 0:
            break
        # If the value comes from the row above, we didn't include this route
        if max_passengers == dp_table[i-1][w]:
            continue
        else:
            # We included this route!
            selected_routes.append(routes[i-1])
            max_passengers -= value[i-1]
            w -= cost[i-1]

    return dp_table[n][total_budget], selected_routes

# 5. Run the Optimization
print(f"\n🚍 Initiating DP Resource Allocation (Total Fleet: {TOTAL_AVAILABLE_BUSES} Buses) 🚍")
max_served, optimal_routes = optimize_fleet_allocation(routes, buses_required, passenger_demand, TOTAL_AVAILABLE_BUSES)

print(f"Maximum Passengers Served: {max_served}")
print("Optimal Routes to Activate:")
for r in optimal_routes:
    # Find the index to print out the specific data for each chosen route
    idx = routes.index(r)
    print(f"- {r} (Requires {buses_required[idx]} buses, Serves {passenger_demand[idx]} passengers)")

Loading Public Transit Demand Data...

🚍 Initiating DP Resource Allocation (Total Fleet: 30 Buses) 🚍
Maximum Passengers Served: 15500
Optimal Routes to Activate:
- Route 4 (Corniche) (Requires 8 buses, Serves 4000 passengers)
- Route 3 (Salah Salem) (Requires 12 buses, Serves 6500 passengers)
- Route 2 (Autostrad) (Requires 10 buses, Serves 5000 passengers)


In [28]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error

print("🤖 Initializing AI Traffic Prediction Module...")

# 1. Load the Traffic Data
df_traffic = pd.read_csv("Traffic_Flow_Data_Patterns.csv")

# 2. Reshape the Data for Machine Learning
# Right now, your times of day are columns. ML needs them to be rows (Features).
# We use 'melt' to unpivot the table.
df_ml = df_traffic.melt(id_vars=['RoadID'],
                        value_vars=['Morning Peak(veh/h)', 'Afternoon (veh/h)', 'Evening Peak(veh/h)', 'Night(veh/h)'],
                        var_name='Time_of_Day',
                        value_name='Traffic_Volume')

# 3. Encode the Text into Numbers
road_encoder = LabelEncoder()
time_encoder = LabelEncoder()

df_ml['Road_Code'] = road_encoder.fit_transform(df_ml['RoadID'])
df_ml['Time_Code'] = time_encoder.fit_transform(df_ml['Time_of_Day'])

# 4. Define our Features (X) and Target to predict (y)
X = df_ml[['Road_Code', 'Time_Code']]
y = df_ml['Traffic_Volume']

# 5. Split data into "Training" and "Testing" sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 6. Train the AI Model!
print("🧠 Training Random Forest ML Model...")
ai_model = RandomForestRegressor(n_estimators=100, random_state=42)
ai_model.fit(X_train, y_train)

# Test how accurate it is
predictions = ai_model.predict(X_test)
error = mean_absolute_error(y_test, predictions)
print(f"✅ Training Complete! The AI's average prediction error is roughly {round(error, 2)} vehicles.")

# 7. Create a clean function for Dijkstra to use
def predict_congestion(road_id, time_of_day):
    """Ask the AI to predict traffic volume for a specific road and time."""
    try:
        # Convert the human text into the AI's secret number codes
        r_code = road_encoder.transform([road_id])[0]
        t_code = time_encoder.transform([time_of_day])[0]

        # Ask the AI to predict!
        # We package the numbers back into a DataFrame with the correct labels
        import pandas as pd
        input_data = pd.DataFrame([[r_code, t_code]], columns=['Road_Code', 'Time_Code'])
        predicted_volume = ai_model.predict(input_data)
        return predicted_volume[0]

    except ValueError:
        # If the AI has never seen this road before (like the brand new one we built!),
        # return a safe, average default guess so the system doesn't crash.
        return 1500

# Let's test the AI!
test_road = '1-3'
test_time = 'Evening Peak(veh/h)'
prediction = predict_congestion(test_road, test_time)
print(f"\n🔮 AI Prediction: Road {test_road} during {test_time} will have {round(prediction)} vehicles.")

🤖 Initializing AI Traffic Prediction Module...
🧠 Training Random Forest ML Model...
✅ Training Complete! The AI's average prediction error is roughly 285.07 vehicles.

🔮 AI Prediction: Road 1-3 during Evening Peak(veh/h) will have 2428 vehicles.
